# Discharges within 24h of an out-of-ICU transfer — run notebook

Same core as the CLI (`src/metric.py`). Use this to run the real export, inspect the matched rows, and chart the daily trend.

**Inputs**
- Raw RWB export from template `Modify ADT Transfers [Location]` (M24047 [100432]) → save to `data/`
- Or the synthetic fixture for a smoke test.

**Rule** (per requester): for each encounter keep the transfer CLOSEST to discharge; count the encounter once if `0 <= Disch − Eff <= 24h`. Timestamps are PDT (America/Los_Angeles).

In [ ]:
import os, sys
from pathlib import Path
# Make the project root importable (this notebook lives in notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('project root:', ROOT)

In [ ]:
from src.config import load_config
cfg = load_config('config/metric.yaml')
print(cfg)

In [ ]:
from src.clean import clean
RAW = 'data/export.xls'   # <-- point at your real export
if not os.path.exists(RAW):
    RAW = 'tests/fixtures/sample_transfer_events.csv'
    print('WARNING: no data/export.xls; using the synthetic fixture for a smoke test')
stats = clean(RAW, 'output/transfer_events.csv', from_units=cfg.from_units, timezone=cfg.timezone)
stats

In [ ]:
import csv, json
from datetime import datetime
from src.metric import TransferEvent, compute

def load_cleaned(path):
    evs = []
    with open(path, newline='', encoding='utf-8') as fh:
        for row in csv.DictReader(fh):
            evs.append(TransferEvent(
                patient_id=row['patient_id'], csn=row['csn'], visit_id=row['visit_id'],
                from_unit=row['from_unit'], to_unit=row['to_unit'],
                eff_ts=datetime.fromisoformat(row['eff_ts']),
                disch_ts=datetime.fromisoformat(row['disch_ts']),
                delta_hours=float(row['delta_hours']),
            ))
    return evs

events = load_cleaned('output/transfer_events.csv')
res = compute(events, cfg)
print(json.dumps({k: v for k, v in res.to_dict().items() if k not in ('matches', 'by_day')}, indent=2))

In [ ]:
import pandas as pd
pd.DataFrame([{
    'patient_id': m.patient_id, 'csn': m.csn, 'from': m.from_unit, 'to': m.to_unit,
    'transfer_ts': m.transfer_ts.isoformat(), 'discharge_ts': m.discharge_ts.isoformat(),
    'delta_hours': round(m.delta_hours, 2),
} for m in res.matches])

In [ ]:
import matplotlib.pyplot as plt
rows = sorted(res.by_day.items())
if rows:
    dates = [d.isoformat() for d, _ in rows]
    within = [v['within'] for _, v in rows]
    total = [v['total'] for _, v in rows]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(dates, within, color='#c0392b', label='within 24h')
    ax.plot(dates, total, color='#2c3e50', marker='o', label='encounters')
    ax.set_title('Discharges within 24h of out-of-ICU transfer')
    ax.set_ylabel('count')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('no qualifying rows')

In [ ]:
from dataclasses import replace
cfg_all = replace(cfg, dedup_rule='count_all')
res_all = compute(events, cfg_all)
print('closest_to_discharge:', res.discharges_within_window, '/', res.encounters_total)
print('count_all:', res_all.discharges_within_window, '/', res_all.total_rows)

## Handoff
- The Python core is authoritative; Power BI/DAX should mirror `sql/02_24h_flag.sql`.
- Confirmed: timezone PDT, same-encounter discharge, dedup = closest-to-discharge.
- Next: drop the real export at `data/export.xls` and re-run top-to-bottom.